# Blood-cell YOLO26 fine-tuning

This private Kaggle notebook fine-tunes a pretrained YOLO26n detector on the supplied YOLO-format blood-smear dataset. It uses the train/validation splits for training and model selection, then evaluates the selected checkpoint on the held-out test split. GPU acceleration is enabled in the Kaggle kernel metadata.

In [ ]:
%pip -q install "ultralytics>=8.4.0" "Pillow>=10.0"

from pathlib import Path
import sys

dataset_mount = Path("/kaggle/input/blood-smear-yolo26-data")
if not dataset_mount.is_dir():
    raise FileNotFoundError(f"Expected Kaggle dataset under {dataset_mount}")
payload = dataset_mount / "dataset_payload.zip"
if (dataset_mount / "images" / "train").is_dir():
    dataset_root = dataset_mount
elif payload.is_file():
    dataset_root = Path("/kaggle/working/input")
    dataset_root.mkdir(parents=True, exist_ok=True)
    import zipfile
    with zipfile.ZipFile(payload) as archive:
        archive.extractall(dataset_root)
else:
    raise FileNotFoundError(f"No YOLO dataset files found under {dataset_mount}")
project_source = dataset_root / "project_src"
if not (dataset_root / "images" / "train").is_dir():
    raise FileNotFoundError(f"Expected YOLO training files under {dataset_root}")
if not project_source.is_dir():
    raise FileNotFoundError(f"Expected project source under {project_source}")
sys.path.insert(0, str(project_source))

from blood_cells.dataset import write_dataset_report
from blood_cells.training import run_training

output_root = Path("/kaggle/working")
write_dataset_report(dataset_root, output_root / "reports" / "dataset_report.json")
performance = run_training(
    dataset_root,
    output_root,
    model_name="yolo26n.pt",
    epochs=100,
    image_size=640,
    batch_size=16,
    patience=20,
    device=0,
)
print("Held-out test metrics:", performance["metrics"])
